# Human PPI: embedding geometry vs experimental essentiality

**Goal:** Check whether **hyperbolic radius** and simple **chart radii** differ between genes labeled **essential** vs **non-essential** in a consolidated human screen dataset (SNAP / OGEE-style table), for **D=2** and **D=3** D-Mercator runs under ``d-mercator-run/d2/`` and ``d3/``.

**Inputs:** ``resources/G-HumanEssential.tsv.gz`` + ``resources/Homo_sapiens.gene_info.gz`` (see ``resources/SOURCES.md``). Run ``python fetch_resources.py`` from this folder if files are missing.

**Working directory:** ``notebooks/validation/`` (or adjust ``REPO`` in the first code cell).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

def _find_repo() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "d-mercator-run").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("Set cwd to repo or notebooks/validation (need d-mercator-run/). cwd=" + str(Path.cwd()))


REPO = _find_repo()
sys.path.insert(0, str(REPO / "notebooks" / "dmercator"))
sys.path.insert(0, str(REPO / "notebooks" / "dmercator3d"))
sys.path.insert(0, str(REPO / "notebooks" / "validation"))

from embedding_essentiality import attach_essentiality, disk_radius_from_ortho_xy, load_snap_essentiality

VAL = REPO / "notebooks" / "validation"
RES = VAL / "resources"
assert (RES / "G-HumanEssential.tsv.gz").is_file(), "Run: python fetch_resources.py"
assert (RES / "Homo_sapiens.gene_info.gz").is_file(), "Run: python fetch_resources.py"

ess = load_snap_essentiality()
print("SNAP rows (symbol-mapped):", len(ess))
print(ess["essential"].value_counts())


## Native **D = 2** (`d-mercator-run/d2/`)

**Geometry:** `Inf.Hyp.Rad` (hyperbolic radial in the model) and **orthographic disk radius** $\sqrt{x^2+y^2}$ after `dmercator_io.ortho_xy_disk` (same chart family as the 2D disk export).

**Statistics:** Mann–Whitney on `Inf.Hyp.Rad`, point-biserial correlation, optional logistic regression controlling for **graph degree**.


In [ ]:
import dmercator_io as dm

paths = dm.paths_for_run("d2")
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
deg_map = {str(a): int(b) for a, b in G.degree()}
df = df.assign(degree=df["Vertex"].astype(str).map(deg_map))
df["degree"] = df["degree"].fillna(0).astype(int)
x, y = dm.ortho_xy_disk(df)
df["r_ortho_disk"] = disk_radius_from_ortho_xy(df, x, y)

m = attach_essentiality(df, ess=ess)
labeled = m[m["essential"].notna()].copy()
labeled["essential"] = labeled["essential"].astype(int)
print("D=2 | vertices", len(m), "| labeled", len(labeled), "| essential", int(labeled["essential"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)
for ax, col, title in zip(
    axes,
    ["Inf.Hyp.Rad", "r_ortho_disk"],
    ["Inf.Hyp.Rad", "Orthographic disk radius sqrt(x^2+y^2)"],
):
    v0 = labeled.loc[labeled["essential"] == 0, col].to_numpy()
    v1 = labeled.loc[labeled["essential"] == 1, col].to_numpy()
    ax.violinplot([v0, v1], positions=[0, 1], showmeans=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["non-essential", "essential"])
    ax.set_ylabel(col)
    ax.set_title(title)
fig.suptitle("Native D=2 run (d2)")
fig.tight_layout()
plt.show()

g0 = labeled.loc[labeled["essential"] == 0, "Inf.Hyp.Rad"]
g1 = labeled.loc[labeled["essential"] == 1, "Inf.Hyp.Rad"]
u_stat, u_p = stats.mannwhitneyu(g1, g0, alternative="two-sided")
print("Mann–Whitney Inf.Hyp.Rad (essential vs non): U=", u_stat, "p=", u_p)

r_pb, r_p = stats.pointbiserialr(labeled["essential"], labeled["Inf.Hyp.Rad"])
print("Point-biserial essential x Inf.Hyp.Rad: r=", r_pb, "p=", r_p)

c_pb, c_p = stats.pointbiserialr(labeled["essential"], labeled["r_ortho_disk"])
print("Point-biserial essential x disk radius: r=", c_pb, "p=", c_p)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xf = labeled[["Inf.Hyp.Rad", "degree"]].to_numpy(dtype=float)
y = labeled["essential"].to_numpy()
Xs = StandardScaler().fit_transform(Xf)
clf = LogisticRegression(max_iter=300)
clf.fit(Xs, y)
print("Logistic coef (scaled): Inf.Hyp.Rad, degree:", clf.coef_[0])


## Native **D = 3** (`d-mercator-run/d3/`)

**Geometry:** `Inf.Hyp.Rad` and **stereographic ball radius** $\|\mathbf{y}\|$ in $\mathbb{R}^3$ from `ball_projection.stereographic_s3_to_r3` on unit directions (north pole chart).

**Same tests** as D=2 for comparability.


In [ ]:
import dmercator3d_io as dm
from ball_projection import stereographic_s3_to_r3

paths = dm.paths_for_run("d3")
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
deg_map = {str(a): int(b) for a, b in G.degree()}
df = df.assign(degree=df["Vertex"].astype(str).map(deg_map))
df["degree"] = df["degree"].fillna(0).astype(int)
U = dm.normalize_direction_nd(df)
x1, x2, x3, x4 = U[:, 0], U[:, 1], U[:, 2], U[:, 3]
Xb, Yb, Zb = stereographic_s3_to_r3(x1, x2, x3, x4, pole="north")
df["r_stereo_ball"] = np.sqrt(Xb * Xb + Yb * Yb + Zb * Zb)

m = attach_essentiality(df, ess=ess)
labeled = m[m["essential"].notna()].copy()
labeled["essential"] = labeled["essential"].astype(int)
print("D=3 | vertices", len(m), "| labeled", len(labeled), "| essential", int(labeled["essential"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)
for ax, col, title in zip(
    axes,
    ["Inf.Hyp.Rad", "r_stereo_ball"],
    ["Inf.Hyp.Rad", "Stereographic R3 ball radius ||y||"],
):
    v0 = labeled.loc[labeled["essential"] == 0, col].to_numpy()
    v1 = labeled.loc[labeled["essential"] == 1, col].to_numpy()
    ax.violinplot([v0, v1], positions=[0, 1], showmeans=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["non-essential", "essential"])
    ax.set_ylabel(col)
    ax.set_title(title)
fig.suptitle("Native D=3 run (d3)")
fig.tight_layout()
plt.show()

g0 = labeled.loc[labeled["essential"] == 0, "Inf.Hyp.Rad"]
g1 = labeled.loc[labeled["essential"] == 1, "Inf.Hyp.Rad"]
u_stat, u_p = stats.mannwhitneyu(g1, g0, alternative="two-sided")
print("Mann–Whitney Inf.Hyp.Rad (essential vs non): U=", u_stat, "p=", u_p)

r_pb, r_p = stats.pointbiserialr(labeled["essential"], labeled["Inf.Hyp.Rad"])
print("Point-biserial essential x Inf.Hyp.Rad: r=", r_pb, "p=", r_p)

c_pb, c_p = stats.pointbiserialr(labeled["essential"], labeled["r_stereo_ball"])
print("Point-biserial essential x stereo ball radius: r=", c_pb, "p=", c_p)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xf = labeled[["Inf.Hyp.Rad", "degree"]].to_numpy(dtype=float)
y = labeled["essential"].to_numpy()
Xs = StandardScaler().fit_transform(Xf)
clf = LogisticRegression(max_iter=300)
clf.fit(Xs, y)
print("Logistic coef (scaled): Inf.Hyp.Rad, degree:", clf.coef_[0])
